In [1]:
import sys
sys.argv = ['']
# -*- coding: utf-8 -*-
"""
Runner do estudo comparativo Pipeline A vs B vs C (Senti-Pred-remake2).

Executa e exporta para CSV/JSON:
  E1.  Reprodução canônica das três pipelines (todos os modelos documentados)
  E2.  Ablação do número de n-grams (what-if principal do usuário)
  E3.  Ablação do tamanho do vocabulário (max_features)
  E4.  Ablação de min_df
  E5.  Ablação de sublinear_tf
  E6.  Toggles de pré-processamento por pipeline
  E7.  Toggles de modelo para a pipeline C (voting hard/soft, class_weight)
  E8.  Fairness de class_weight (balanced aplicado a A/B/C)
  E9.  Cross-pipeline: pré-processamento de cada pipeline com vetorizadores dos outros

Uso:
    python run_abc_comparison.py [--stages 1 2 3 ...] [--out DIR]
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.base import clone

sys.path.insert(0, str(Path('d:\\mlops-experiments\\experiments\\nlp\\twitter-entity-sentiment\\pipelines_abc_comparison\\run_abc_comparison.py').parent))
from pipelines_abc_core import (
    CLEANERS, VEC_CANONICAL, MODELS_A, MODELS_B, MODELS_C,
    apply_cleaner, clean_a, clean_b, clean_c, evaluate, load_data,
    make_linear_svc_c19, make_voting_c, VALID_SENTIMENTS,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier


def _calibrated_svc():
    """LinearSVC com proba (calibração Platt) p/ suportar voting='soft'."""
    return CalibratedClassifierCV(
        LinearSVC(C=0.5, max_iter=3000, dual='auto', random_state=42, tol=1e-5,
                  class_weight='balanced'),
        cv=3, method='sigmoid')

SEED = 42


def champion_model(pipeline):
    """Melhor modelo documentado de cada pipeline para as ablações de features."""
    if pipeline == 'A':
        return 'LinearSVC_C19', make_linear_svc_c19()
    if pipeline == 'B':
        return 'LinearSVC_C19', make_linear_svc_c19()
    return 'Voting_svc_lr', make_voting_c()


# ---------------------------------------------------------------- helpers ----
def abbreviate(row_dict):
    """Copia a linha e serializa campos não-essenciais/objetos grandes."""
    row = {k: v for k, v in row_dict.items()
           if k not in ('y_true', 'y_pred', 'confusion_matrix', 'report')}
    return row


def run_single(pipe, tr, va, vec_params, model_name, model, tag, **extra_meta):
    res = evaluate(pipe, vec_params, model, tr, va, model_name)
    row = abbreviate(res)
    row.update({'tag': tag, **extra_meta})
    return row


# ------------------------------------------------------------------- E1 ------
def e1_canonical(tr, va):
    print('\n[E1] Reprodução canônica das três pipelines ...')
    rows = []
    # Pipeline A
    tr_a = apply_cleaner(tr, clean_a); va_a = apply_cleaner(va, clean_a)
    for name, mk in MODELS_A.items():
        rows.append(run_single('A', tr_a, va_a, VEC_CANONICAL['A'], name, mk(),
                               'canonical', cleaner='clean_a'))
    # Pipeline B
    tr_b = apply_cleaner(tr, clean_b); va_b = apply_cleaner(va, clean_b)
    for name, mk in MODELS_B.items():
        rows.append(run_single('B', tr_b, va_b, VEC_CANONICAL['B'], name, mk(),
                               'canonical', cleaner='clean_b'))
    # Pipeline C
    tr_c = apply_cleaner(tr, clean_c); va_c = apply_cleaner(va, clean_c)
    for name, mk in MODELS_C.items():
        rows.append(run_single('C', tr_c, va_c, VEC_CANONICAL['C'], name, mk(),
                               'canonical', cleaner='clean_c'))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E2 ------
NG_RAMPS = [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 2), (2, 3)]


def e2_ngrams(tr, va):
    print('\n[E2] Ablação do nº de n-grams (max N) por pipeline ...')
    rows = []
    for pipe in ['A', 'B', 'C']:
        cleaner = CLEANERS[pipe]
        tr_p = apply_cleaner(tr, cleaner); va_p = apply_cleaner(va, cleaner)
        m_name, model = champion_model(pipe)
        for (lo, hi) in NG_RAMPS:
            vp = dict(VEC_CANONICAL[pipe]); vp['ngram_range'] = (lo, hi)
            rows.append(run_single(pipe, tr_p, va_p, vp, m_name, model, 'ablation_ngram',
                                   ngram_lo=lo, ngram_hi=hi, default_ngram=str(VEC_CANONICAL[pipe]['ngram_range'])))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E3 ------
MAX_FEATS = [10000, 25000, 50000, 70000, 100000, 150000, 200000]


def e3_max_features(tr, va):
    print('\n[E3] Ablação do tamanho do vocabulário (max_features) ...')
    rows = []
    for pipe in ['A', 'B', 'C']:
        cleaner = CLEANERS[pipe]
        tr_p = apply_cleaner(tr, cleaner); va_p = apply_cleaner(va, cleaner)
        m_name, model = champion_model(pipe)
        for mf in MAX_FEATS:
            vp = dict(VEC_CANONICAL[pipe]); vp['max_features'] = mf
            rows.append(run_single(pipe, tr_p, va_p, vp, m_name, model, 'ablation_max_features',
                                   max_features=mf, default_max_features=VEC_CANONICAL[pipe]['max_features']))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E4 ------
MIN_DFS = [1, 2, 3, 5]


def e4_min_df(tr, va):
    print('\n[E4] Ablação de min_df ...')
    rows = []
    for pipe in ['A', 'B', 'C']:
        cleaner = CLEANERS[pipe]
        tr_p = apply_cleaner(tr, cleaner); va_p = apply_cleaner(va, cleaner)
        m_name, model = champion_model(pipe)
        for md in MIN_DFS:
            vp = dict(VEC_CANONICAL[pipe]); vp['min_df'] = md
            rows.append(run_single(pipe, tr_p, va_p, vp, m_name, model, 'ablation_min_df',
                                   min_df=md, default_min_df=VEC_CANONICAL[pipe]['min_df']))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E5 ------
def e5_sublinear(tr, va):
    print('\n[E5] Ablação de sublinear_tf ...')
    rows = []
    for pipe in ['A', 'B', 'C']:
        cleaner = CLEANERS[pipe]
        tr_p = apply_cleaner(tr, cleaner); va_p = apply_cleaner(va, cleaner)
        m_name, model = champion_model(pipe)
        for sl in [True, False]:
            vp = dict(VEC_CANONICAL[pipe]); vp['sublinear_tf'] = sl
            rows.append(run_single(pipe, tr_p, va_p, vp, m_name, model, 'ablation_sublinear_tf',
                                   sublinear_tf=sl, default_sublinear_tf=VEC_CANONICAL[pipe]['sublinear_tf']))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E6 ------
# Para cada pipeline, variações de toggles de pré-processamento (1 toggle p/ vez)
PREPROC_ABLATIONS = {
    'A': [
        ('A_default', dict()),
        ('A_keep_hashtags', dict(keep_hashtags=True)),
        ('A_keep_punct', dict(keep_punct=True)),
        ('A_keep_digits', dict(keep_digits=True)),
        ('A_keep_punct_digits', dict(keep_punct=True, keep_digits=True)),
    ],
    'B': [
        ('B_default', dict()),
        ('B_drop_hashtags', dict(drop_hashtags=True)),
        ('B_drop_punct', dict(drop_punct=True)),
        ('B_drop_digits', dict(drop_digits=True)),
    ],
    'C': [
        ('C_default', dict()),
        ('C_keep_stopwords', dict(remove_stopwords=False)),
        ('C_no_lemmatize', dict(lemmatize=False)),
        ('C_no_contractions', dict(expand_contractions=False)),
        ('C_drop_exclam_question', dict(keep_question_mark=False)),
        ('C_remove_hashtag_word', dict(keep_hashtag_word=False)),
    ],
}


def e6_preprocessing(tr, va):
    print('\n[E6] Toggles de pré-processamento por pipeline ...')
    rows = []
    for pipe, variants in PREPROC_ABLATIONS.items():
        m_name, model = champion_model(pipe)
        for label, kwargs in variants:
            cleaner = CLEANERS[pipe]
            tr_p = apply_cleaner(tr, cleaner, **kwargs)
            va_p = apply_cleaner(va, cleaner, **kwargs)
            rows.append(run_single(pipe, tr_p, va_p, VEC_CANONICAL[pipe], m_name, model,
                                   'ablation_preprocessing', cleaner_label=label, **kwargs))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E7 ------
def e7_model_c(tr, va):
    print('\n[E7] Ablação de modelo da pipeline C ...')
    tr_c = apply_cleaner(tr, clean_c); va_c = apply_cleaner(va, clean_c)
    vp = VEC_CANONICAL['C']
    variants = {
        'LinearSVC_C0.5': lambda: LinearSVC(C=0.5, max_iter=3000, dual='auto',
                                            random_state=42, tol=1e-5, class_weight='balanced'),
        'LinearSVC_C0.5_noweight': lambda: LinearSVC(C=0.5, max_iter=3000, dual='auto',
                                                     random_state=42, tol=1e-5),
        'LinearSVC_C19': make_linear_svc_c19,
        'LogReg_C10_balanced': lambda: LogisticRegression(C=10, max_iter=1000, solver='lbfgs',
                                                          multi_class='multinomial', random_state=42,
                                                          class_weight='balanced'),
        'LogReg_C10': lambda: LogisticRegression(C=10, max_iter=1000, solver='lbfgs',
                                                 multi_class='multinomial', random_state=42),
        'Voting_hard': make_voting_c,
        'Voting_soft': lambda: VotingClassifier(
            estimators=[('svc', _calibrated_svc()),
                        ('lr', LogisticRegression(C=10, max_iter=1000, solver='lbfgs',
                                                  multi_class='multinomial', random_state=42,
                                                  class_weight='balanced'))],
            voting='soft'),
        'Voting_hard_noweight': lambda: VotingClassifier(
            estimators=[('svc', LinearSVC(C=0.5, max_iter=3000, dual='auto', random_state=42, tol=1e-5)),
                        ('lr', LogisticRegression(C=10, max_iter=1000, solver='lbfgs',
                                                  multi_class='multinomial', random_state=42))],
            voting='hard'),
    }
    rows = []
    for name, mk in variants.items():
        rows.append(run_single('C', tr_c, va_c, vp, name, mk(), 'ablation_model'))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E8 ------
def e8_balanced_fairness(tr, va):
    """Mesmo pipeline/features, mas com class_weight='balanced' nos lineares (fairness)."""
    print('\n[E8] Fairness: class_weight=balanced nos modelos lineares de A/B/C ...')
    rows = []
    m = lambda: LogisticRegression(C=10, max_iter=1000, solver='lbfgs',
                                   multi_class='multinomial', random_state=42,
                                   class_weight='balanced')
    svm = lambda: LinearSVC(C=19.0, max_iter=20000, random_state=42, class_weight='balanced')
    variants = {
        'A_balanced_LR_C10': 'A', 'A_balanced_SVC_C19': 'A',
        'B_balanced_LR_C10': 'B', 'B_balanced_SVC_C19': 'B',
        'C_balanced_LR_C10': 'C', 'C_balanced_SVC_C19': 'C',
    }
    for label, pipe in variants.items():
        cleaner = CLEANERS[pipe]
        tr_p = apply_cleaner(tr, cleaner); va_p = apply_cleaner(va, cleaner)
        model = m() if 'LR' in label else svm()
        rows.append(run_single(pipe, tr_p, va_p, VEC_CANONICAL[pipe], label.replace('_balanced_', '_'), model,
                               'fairness_balanced'))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E9 ------
def e9_cross_pp(tr, va):
    """Cross: cada cleaner aplicado sob vetorizadores das outras pipelines + canônicos."""
    print('\n[E9] Cross pré-processamento × vetorizador ...')
    rows = []
    for pp in ['A', 'B', 'C']:
        cleaner = CLEANERS[pp]
        tr_p = apply_cleaner(tr, cleaner); va_p = apply_cleaner(va, cleaner)
        m_name, model = champion_model(pp)
        # usa o modelo campeão da própria pipeline sob os 3 vetorizadores canônicos
        for vec_owner in ['A', 'B', 'C']:
            rows.append(run_single(f'PP={pp}', tr_p, va_p, VEC_CANONICAL[vec_owner], m_name, model,
                                   'cross_pp_x_vec', cleaner=pp, vectorizer_owner=vec_owner))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- E10 -----
def e10_cv_evaluation(tr, va):
    print('\n[E10] CV-5 evaluation para as pipelines campeãs ...')
    rows = []
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    configs = [
        ('A', 'ExtraTrees', CLEANERS['A'], VEC_CANONICAL['A'], MODELS_A['ExtraTrees']()),
        ('B', 'LinearSVC_C19', CLEANERS['B'], VEC_CANONICAL['B'], make_linear_svc_c19()),
        ('C', 'Voting_svc_lr', CLEANERS['C'], VEC_CANONICAL['C'], make_voting_c()),
        ('C', 'LinearSVC_C0.5', CLEANERS['C'], VEC_CANONICAL['C'], MODELS_C['LinearSVC_C0.5']()),
    ]
    
    for pipe, m_name, cleaner, vec_params, model in configs:
        print(f"  -> CV-5 para Pipeline {pipe} com {m_name}")
        
        with mlflow.start_run(nested=True):
            mlflow.log_param("stage", "cv_evaluation")
            mlflow.log_param("pipeline", pipe)
            mlflow.log_param("model_name", m_name)
            
            tr_p = apply_cleaner(tr, cleaner)
            X_clean = np.array(tr_p['clean'].tolist())
            y_clean = np.array(tr_p['sentiment'].tolist())
            
            f1_scores = []
            
            va_p = apply_cleaner(va, cleaner)
            holdout_res = evaluate(pipe, vec_params, model, tr_p, va_p, m_name, extra_meta={'stage': 'cv_holdout'})
            val_f1_macro = holdout_res['f1_macro']
            
            pipeline_obj = Pipeline([
                ('vectorizer', TfidfVectorizer(**vec_params)),
                ('classifier', model)
            ])
            pipeline_obj.fit(X_clean, y_clean)
            train_preds = pipeline_obj.predict(X_clean)
            train_f1_macro = f1_score(y_clean, train_preds, average='macro')
            
            for fold, (train_idx, test_idx) in enumerate(cv.split(X_clean, y_clean)):
                X_tr, X_te = X_clean[train_idx], X_clean[test_idx]
                y_tr, y_te = y_clean[train_idx], y_clean[test_idx]
                
                pipe_obj = Pipeline([
                    ('vectorizer', TfidfVectorizer(**vec_params)),
                    ('classifier', clone(model))
                ])
                pipe_obj.fit(X_tr, y_tr)
                preds = pipe_obj.predict(X_te)
                f1 = f1_score(y_te, preds, average='macro')
                f1_scores.append(f1)
                mlflow.log_metric(f"fold_{fold}_f1", f1)
            
            f1_cv_mean = np.mean(f1_scores)
            f1_cv_std = np.std(f1_scores)
            
            overfit_gap = train_f1_macro - val_f1_macro
            shift_gap = val_f1_macro - f1_cv_mean
            
            mlflow.log_metric("cv_f1_macro_mean", f1_cv_mean)
            mlflow.log_metric("cv_f1_macro_std", f1_cv_std)
            mlflow.log_metric("train_f1_macro", train_f1_macro)
            mlflow.log_metric("val_f1_macro", val_f1_macro)
            mlflow.log_metric("overfit_gap", overfit_gap)
            mlflow.log_metric("shift_gap", shift_gap)
            
            rows.append({
                'pipeline': pipe,
                'model': m_name,
                'cv_f1_mean': f1_cv_mean,
                'cv_f1_std': f1_cv_std,
                'train_f1': train_f1_macro,
                'val_f1': val_f1_macro,
                'overfit_gap': overfit_gap,
                'shift_gap': shift_gap
            })
            
    return pd.DataFrame(rows)


# ================================================================ driver ======
STAGES = {
    'canonical': e1_canonical,
    'ngrams': e2_ngrams,
    'max_features': e3_max_features,
    'min_df': e4_min_df,
    'sublinear_tf': e5_sublinear,
    'preprocessing': e6_preprocessing,
    'model_c': e7_model_c,
    'fairness': e8_balanced_fairness,
    'cross': e9_cross_pp,
    'cv': e10_cv_evaluation,
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--stages', nargs='*', default=list(STAGES.keys()))
    ap.add_argument('--out', default=None, help='diretório de saída (artifacts)')
    ap.add_argument('--seed', type=int, default=42)
    args = ap.parse_args([])

    t0 = time.time()
    tr, va = load_data()
    print(f'Dados carregados: train={len(tr)} val={len(va)}  (seed={args.seed})')

    mlflow.set_experiment("senti-pred-abc-comparison")

    out_dir = Path(args.out) if args.out else Path('artifacts_abc')
    out_dir.mkdir(parents=True, exist_ok=True)

    meta = {'seed': args.seed, 'train_rows': len(tr), 'val_rows': len(va),
            'executed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
            'stages': list(args.stages)}

    all_frames = []
    for stage in args.stages:
        fn = STAGES[stage]
        s0 = time.time()
        df = fn(tr, va)
        df['stage'] = stage
        df.to_csv(out_dir / f'results_{stage}.csv', index=False)
        all_frames.append(df)
        print(f'  -> {stage}: {len(df)} runs em {time.time() - s0:.1f}s')

    merged = pd.concat(all_frames, ignore_index=True)
    merged.to_csv(out_dir / 'results_all.csv', index=False)

    with open(out_dir / 'meta.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

    print(f'\nTotal de execuções: {len(merged)}  em {time.time() - t0:.1f}s')
    print(f'Artefatos salvos em: {out_dir.resolve()}')


if __name__ == '__main__':
    main()

Dados carregados: train=73996 val=1000  (seed=42)


2026/08/12 18:33:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/08/12 18:33:37 INFO mlflow.store.db.utils: Updating database tables


2026/08/12 18:33:37 INFO alembic.runtime.migration: Context impl SQLiteImpl.


2026/08/12 18:33:37 INFO alembic.runtime.migration: Will assume non-transactional DDL.


...

C:\Users\pedro\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


C:\Users\pedro\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  -> CV-5 para Pipeline C com LinearSVC_C0.5


2026/08/12 19:32:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  -> cv: 4 runs em 948.4s

Total de execuções: 118  em 3600.1s
Artefatos salvos em: D:\mlops-experiments\experiments\nlp\twitter-entity-sentiment\pipelines_abc_comparison\artifacts_abc
